# Chimera v2.4 — UMAP integration visualisation

The 32-gate validation in
[`10_paper_validation.ipynb`](10_paper_validation.ipynb) relies on per-gene
metrics (α dynamics, residual magnitude, adversarial trajectory). Those are
the *decision criteria* — they're what the training loop optimises for. This
notebook adds the **visual integration test** a reader expects from a
single-cell integration paper: side-by-side UMAPs of (a) the raw,
un-integrated counts and (b) the Chimera latent **z**, coloured by species
and cell type.

**Design choices.**
- **No scib.** Integration metrics here are plain kNN purity / entropy +
  silhouette (subsampled). Each runs in seconds. This matches the
  workflow's "fast diagnostics only" rule.
- **Fair baseline.** The pre-integration UMAP uses *log1p(CPM)* of the same
  count matrix Chimera sees. PCA(50) first for speed; same UMAP hyperparameters
  for both baselines and z.
- **Caveat 34 honest framing.** Chimera's design makes z nonlinearly species-
  aware (encoder receives `species_oh`). The linear adversarial classifier in
  training reaches near-chance, but a nonlinear probe can still recover
  species. This notebook shows the honest trade-off: z sharpens cell-type
  biology but does not fully dissolve species structure unless the
  `mask_species_in_encoder` + `mask_batch_in_encoder` flags are set (iter24
  recipe). If you need a fully species-invariant representation, use
  [`ChimeraModel.get_shared_only_expression`](../chimera/_model.py)
  — the shared decoder output is architecturally species-free.

**Sections.**
1. Pre-integration baseline (UMAP of raw log-counts).
2. Chimera z — UMAP on the default run.
2b. Chimera shared-decoder expression (`mu_shared_only`) — UMAP. This is
    architecturally species-free per round2 Caveat 34 and is the correct
    integration-success target when z retains nonlinear species structure.
3. Optional: Chimera z with species+batch masks (iter24 variant).
4. Integration scorecard (fast, quantitative) across raw / z / mu_shared.
5. Per-species neighbour composition (which species does each species' cells
   neighbour? Should be uniform for good integration).
6. Interpretation.

**Default runs.** `RUN_NAME = "iter16-slow-classifier"` (Human + Monkey,
2-species, 32/32 gates). `COMPARE_RUN = "iter24-mask-both"` for the
species-invariant-z variant. Change them in the next cell to validate a
different run.


In [ ]:
import sys, time, warnings
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import _paper_utils as pu
pu.set_paper_style()

RUN_NAME = "iter16-slow-classifier"
COMPARE_RUN = "iter24-mask-both"  # set to None to skip §3
SAVE_FIGS = False
SEED = 0
UMAP_CELLS = 6000   # subsample cap for UMAP — 6k is fast and shows structure
SCORECARD_CELLS = 8000
K_NN = 15

run = pu.load_run(RUN_NAME)
adata = pu.load_adata_for_run(run)

fs = run["final_state"]
z = fs["z_mean"]                       # (N, n_latent)
species_labels = fs["species_labels"]  # (N,)
n_species = int(fs["n_species"])
species_names = list(adata.obs["species"].astype("category").cat.categories)
spcolors = pu.species_colors(species_names)

if "cell_type" in adata.obs.columns:
    ct_series = adata.obs["cell_type"].astype("category")
    cell_type_names = list(ct_series.cat.categories)
    celltype_labels = ct_series.cat.codes.values
else:
    cell_type_names = []
    celltype_labels = None

rng = np.random.default_rng(SEED)
if adata.n_obs > UMAP_CELLS:
    umap_idx = rng.choice(adata.n_obs, size=UMAP_CELLS, replace=False)
    umap_idx.sort()
else:
    umap_idx = np.arange(adata.n_obs)

print(f"run={RUN_NAME}  adata={adata.shape}")
print(f"species (K={n_species}): {species_names}")
print(f"cell types (n={len(cell_type_names)})")
print(f"UMAP subsample: {len(umap_idx)} cells")


## §1 — Pre-integration baseline (raw log-counts UMAP)

The baseline is the standard scanpy-style starting point: size-factor
normalise each cell to 10 000 counts, log1p, PCA(50), UMAP. **No batch or
species correction.** A good integration method (Chimera §2 below) should
move from "species clusters dominate" here to "cell-type clusters dominate"
there — without collapsing biology.


In [ ]:
t0 = time.time()
X_log, kept = pu.lognorm_from_counts(adata, max_cells=UMAP_CELLS, seed=SEED)
umap_idx = kept  # lognorm returned the cells it actually used
U_raw = pu.fit_umap(X_log, n_neighbors=15, min_dist=0.3, pca_first=50, seed=SEED)
print(f"raw UMAP: {X_log.shape} in {time.time()-t0:.1f}s")

sub_species = species_labels[umap_idx]
sub_ct = celltype_labels[umap_idx] if celltype_labels is not None else None

def scatter_by_label(ax, U, labels, names, colors_or_cmap, title, ms=3, alpha=0.6):
    if isinstance(colors_or_cmap, dict):
        for i, name in enumerate(names):
            sel = labels == i
            ax.scatter(U[sel, 0], U[sel, 1], s=ms, alpha=alpha,
                       color=colors_or_cmap[name], label=name, linewidths=0)
        ax.legend(loc="best", markerscale=2.5, fontsize=8)
    else:
        cmap = plt.get_cmap(colors_or_cmap)
        for i, name in enumerate(names):
            sel = labels == i
            ax.scatter(U[sel, 0], U[sel, 1], s=ms, alpha=alpha,
                       color=cmap(i / max(len(names) - 1, 1)),
                       label=str(name)[:18], linewidths=0)
        if len(names) <= 12:
            ax.legend(loc="best", markerscale=2.5, fontsize=7, ncol=2)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
scatter_by_label(axes[0], U_raw, sub_species, species_names, spcolors,
                 "Raw log-counts — species")
if sub_ct is not None:
    scatter_by_label(axes[1], U_raw, sub_ct, cell_type_names, "tab20",
                     "Raw log-counts — cell type")
else:
    axes[1].set_axis_off()
fig.suptitle("§1 · Pre-integration UMAP (baseline)", y=1.02)
if SAVE_FIGS:
    fig.savefig(f"figs/nb12_{RUN_NAME}_raw_umap.png", bbox_inches="tight")
plt.show()


## §2 — Chimera z — integrated latent UMAP

The same cells as §1, but now embedded from the Chimera encoder's `z`. The
encoder took species and batch one-hots as additional inputs and was
adversarially pressured to drop species information from `z`. Observe:

- **Species panel**: the coloured clouds should overlap more than in §1
  (species no longer dominates layout).
- **Cell-type panel**: clusters should be sharper and more consistent
  across species (shared biology emerges).

Quantitative versions of these observations are the **integration scorecard
in §4**. A panel-level visual disagreement with the scorecard is itself a
diagnostic signal — the scorecard wins if the two disagree.


In [ ]:
t0 = time.time()
z_sub = z[umap_idx]
U_z = pu.fit_umap(z_sub, n_neighbors=15, min_dist=0.3, pca_first=None, seed=SEED)
print(f"z UMAP: {z_sub.shape} in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
scatter_by_label(axes[0], U_z, sub_species, species_names, spcolors,
                 f"Chimera z ({RUN_NAME}) — species")
if sub_ct is not None:
    scatter_by_label(axes[1], U_z, sub_ct, cell_type_names, "tab20",
                     f"Chimera z ({RUN_NAME}) — cell type")
else:
    axes[1].set_axis_off()
fig.suptitle("§2 · Chimera z — integrated latent", y=1.02)
if SAVE_FIGS:
    fig.savefig(f"figs/nb12_{RUN_NAME}_z_umap.png", bbox_inches="tight")
plt.show()


## §2b — Chimera shared-decoder expression (two variants)

The shared decoder takes only `z` as input (no `species_oh`, no `batch_oh`),
so the `output_shared` head is species-agnostic *by construction*. But the
public [`ChimeraModel.get_shared_only_expression`](../chimera/_model.py)
**adds the per-batch offset** on top of `log_rate_shared` before the
softplus. On datasets where `batch → species` is 1:1 (E1.4 invariant — true
for CAMEX, van Zyl, and every current Chimera dataset), the batch offset
is itself a species offset, so it re-injects species into the output.

This cell therefore compares two variants:

- **`mu_shared_only`** — `exp(log_rate_shared + log_rate_batch) * lib` —
  the public API. With the per-batch offset. **Not species-free.**
- **`mu_shared_cellstate`** — `exp(log_rate_shared) * lib` — without
  the per-batch offset. Species leakage here comes only from `z`
  (through the species-agnostic decoder).

The gap between the two quantifies the **batch-offset channel** of
species leakage. The residual gap between `mu_shared_cellstate` and
true species-invariance measures the **z-leakage channel** (Caveat 34).


In [ ]:
# Reload the trained model and compute TWO variants of mu_shared:
#   (a) mu_shared_only     — with the per-batch offset (public API).
#   (b) mu_shared_cellstate — without the per-batch offset (raw decoder).
# Because batch → species is 1:1 (E1.4 invariant) on this data, the per-batch
# offset carries species by construction — this cell empirically teases apart
# the two species-leakage channels (z carries species; batch offset carries species).
import torch
import torch.nn.functional as F
from scvi import REGISTRY_KEYS
from chimera._model import ChimeraModel, _SPECIES_REGISTRY_KEY

model = ChimeraModel.load(str(run["run_dir"] / "model"), adata=adata)
module = model.module
module.eval()
device = next(module.parameters()).device

def _shared_variants(loader):
    ms_with = []
    ms_no_batch = []
    with torch.no_grad():
        for tensors in loader:
            x = tensors[REGISTRY_KEYS.X_KEY].to(device)
            b_idx = tensors[REGISTRY_KEYS.BATCH_KEY].squeeze(-1).long().to(device)
            s_idx = tensors[_SPECIES_REGISTRY_KEY].squeeze(-1).long().to(device)
            sp_oh = F.one_hot(s_idx, module.n_species).float()
            b_oh = F.one_hot(b_idx, module.n_batch).float()
            h_enc = module.encoder(module._encoder_input(x, sp_oh, b_oh))
            z_mean = module.z_mean_linear(h_enc)
            h = module.decoder_hidden(z_mean)
            log_rate_shared = module.output_shared(h)
            constrained_offset = module.batch_offset()
            log_rate_batch = constrained_offset[b_idx]
            lib = x.sum(-1, keepdim=True).clamp(min=1.0)
            ms_with.append((torch.exp(log_rate_shared + log_rate_batch) * lib).cpu().numpy())
            ms_no_batch.append((torch.exp(log_rate_shared) * lib).cpu().numpy())
    return np.concatenate(ms_with, 0), np.concatenate(ms_no_batch, 0)

t0 = time.time()
loader = model._make_data_loader(adata=adata, batch_size=512, shuffle=False)
mu_shared_batch, mu_shared_cellstate = _shared_variants(loader)
print(f"mu_shared variants shape={mu_shared_batch.shape}  in {time.time()-t0:.1f}s")

def _log1p_cpm(M):
    lib = M.sum(axis=1, keepdims=True)
    lib = np.where(lib > 0, lib, 1.0)
    return np.log1p(M / lib * 1e4)

mu_shared_ln = _log1p_cpm(mu_shared_batch)       # public API (w/ batch offset)
mu_cellstate_ln = _log1p_cpm(mu_shared_cellstate) # cell-state only (no batch)

umap_idx_ms = umap_idx if mu_shared_ln.shape[0] >= umap_idx.max() + 1 else np.arange(min(UMAP_CELLS, mu_shared_ln.shape[0]))
X_ms = mu_shared_ln[umap_idx_ms]
X_cs = mu_cellstate_ln[umap_idx_ms]

t0 = time.time()
U_ms = pu.fit_umap(X_ms, n_neighbors=15, min_dist=0.3, pca_first=50, seed=SEED)
U_cs = pu.fit_umap(X_cs, n_neighbors=15, min_dist=0.3, pca_first=50, seed=SEED)
print(f"mu_shared UMAPs in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
scatter_by_label(axes[0,0], U_ms, species_labels[umap_idx_ms], species_names,
                 spcolors, "mu_shared_only (w/ batch) — species")
scatter_by_label(axes[0,1], U_cs, species_labels[umap_idx_ms], species_names,
                 spcolors, "mu_shared_cellstate (no batch) — species")
if celltype_labels is not None:
    scatter_by_label(axes[1,0], U_ms, celltype_labels[umap_idx_ms],
                     cell_type_names, "tab20",
                     "mu_shared_only (w/ batch) — cell type")
    scatter_by_label(axes[1,1], U_cs, celltype_labels[umap_idx_ms],
                     cell_type_names, "tab20",
                     "mu_shared_cellstate (no batch) — cell type")
else:
    axes[1,0].set_axis_off(); axes[1,1].set_axis_off()
fig.suptitle("§2b · Shared-decoder output — with vs without batch offset", y=1.00)
if SAVE_FIGS:
    fig.savefig(f"figs/nb12_{RUN_NAME}_mu_shared_umap.png", bbox_inches='tight')
plt.show()


## §3 — Chimera z (species+batch masked encoder, iter24 variant)

*Optional*. Skipped when `COMPARE_RUN` is `None` or not found. The iter24
recipe zeroes `species_oh` and `batch_oh` in the encoder input. On this
dataset (batch→species is 1:1), this is the strongest available knob for
a *linearly* species-invariant `z` — it halves the post-hoc linear probe
accuracy (workflow.md §"Caveat 34 nonlinear z-leakage"). Expect:

- species coloring in z to overlap even more than in §2 (sometimes visibly
  "uniform noise" rather than clustered)
- cell-type structure preserved within numeric noise of §2


In [ ]:
compare_avail = False
if COMPARE_RUN is not None:
    try:
        run_c = pu.load_run(COMPARE_RUN)
        z_c = run_c["final_state"]["z_mean"][umap_idx]
        compare_avail = True
    except FileNotFoundError:
        print(f"(COMPARE_RUN={COMPARE_RUN!r} not found; skipping §3)")

if compare_avail:
    t0 = time.time()
    U_zc = pu.fit_umap(z_c, n_neighbors=15, min_dist=0.3, pca_first=None, seed=SEED)
    print(f"z-compare UMAP: {z_c.shape} in {time.time()-t0:.1f}s")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    scatter_by_label(axes[0], U_zc, sub_species, species_names, spcolors,
                     f"Chimera z ({COMPARE_RUN}) — species")
    if sub_ct is not None:
        scatter_by_label(axes[1], U_zc, sub_ct, cell_type_names, "tab20",
                         f"Chimera z ({COMPARE_RUN}) — cell type")
    else:
        axes[1].set_axis_off()
    fig.suptitle("§3 · Chimera z — species+batch-masked encoder", y=1.02)
    if SAVE_FIGS:
        fig.savefig(f"figs/nb12_{COMPARE_RUN}_z_umap.png", bbox_inches='tight')
    plt.show()


## §4 — Integration scorecard (fast, quantitative)

A four-metric scorecard computed on the same cells as above:

| Metric | What it measures | Better |
|---|---|---|
| `knn_species_purity` | mean fraction of 15-NN neighbours with the **same species** | **lower** |
| `knn_species_entropy` | mean normalised Shannon entropy of 30-NN species labels | **higher** |
| `knn_celltype_purity` | mean fraction of 15-NN with the **same cell type** | **higher** |
| `silhouette_celltype` | subsampled (n=5000) silhouette on cell-type labels | **higher** |

We run it on up to four representations:
- `raw_X` — the log1p(CPM) baseline (no integration)
- `z` — the default Chimera run's latent
- `mu_shared_only` — the architecturally species-free shared-decoder output
  (§2b). This is the integration-success target on close-species data.
- `z (masked)` — the species+batch-masked Chimera run's latent (when available)

**Read the scorecard, not the UMAP.** UMAP layouts can be misleading;
kNN-based metrics report the *neighbourhood* structure directly.


In [ ]:
# Use a single shared subset of cells so the three embeddings are compared on
# the same data. We reuse umap_idx (already capped to UMAP_CELLS).
sp_s = species_labels[umap_idx]
ct_s = celltype_labels[umap_idx] if celltype_labels is not None else None

rows = []
sc_raw = pu.integration_scorecard(X_log, sp_s, ct_s,
                                  k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
sc_raw["representation"] = "raw log-counts"
rows.append(sc_raw)

sc_z = pu.integration_scorecard(z[umap_idx], sp_s, ct_s,
                                k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
sc_z["representation"] = f"z ({RUN_NAME})"
rows.append(sc_z)

# mu_shared_only (w/ batch offset — public API)
sc_ms = pu.integration_scorecard(
    X_ms, species_labels[umap_idx_ms],
    celltype_labels[umap_idx_ms] if celltype_labels is not None else None,
    k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
sc_ms["representation"] = "mu_shared_only (w/ batch)"
rows.append(sc_ms)

# mu_shared_cellstate (no batch offset — decoder cell-state only)
sc_cs = pu.integration_scorecard(
    X_cs, species_labels[umap_idx_ms],
    celltype_labels[umap_idx_ms] if celltype_labels is not None else None,
    k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
sc_cs["representation"] = "mu_shared_cellstate (no batch)"
rows.append(sc_cs)

if compare_avail:
    sc_zc = pu.integration_scorecard(run_c["final_state"]["z_mean"][umap_idx],
                                     sp_s, ct_s,
                                     k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
    sc_zc["representation"] = f"z ({COMPARE_RUN})"
    rows.append(sc_zc)

sc_df = pd.DataFrame(rows).set_index("representation")
print(sc_df.round(3).to_string())


In [ ]:
# Bar chart — each metric normalised so that "better" is rightward/up.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, col, better in zip(axes, list(sc_df.columns),
                           ["lower", "higher", "higher", "higher"]):
    vals = sc_df[col].values
    ax.barh(np.arange(len(vals)), vals, color=["#999","#0072B2","#009E73"][:len(vals)])
    ax.set_yticks(np.arange(len(vals)))
    ax.set_yticklabels(sc_df.index, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(col)
    ax.set_title(f"{col}  ({better} = better)", fontsize=9)
fig.suptitle("§4 · Integration scorecard", y=1.04)
if SAVE_FIGS:
    fig.savefig(f"figs/nb12_{RUN_NAME}_scorecard.png", bbox_inches='tight')
plt.show()


## §5 — Per-species neighbour composition

For a well-integrated `z`, if we pick a cell from species *A* and look at its
*k* nearest z-neighbours, the fraction of neighbours from each species
should be close to that species' dataset frequency (null model: uniform
mixing). The matrix below tabulates, for each query species (row), the mean
distribution of its neighbours' species (columns).

- **Diagonal dominance** (near 1 on the diagonal) = species still cluster in
  z, like in raw X.
- **Row matches dataset frequency** = well-mixed z.

This is the same information the scorecard condenses into two scalars, but
broken out per species so asymmetric mixing (e.g. Zebrafish isolates itself
while Human + Monkey + Mouse mix) is visible.


In [ ]:
from sklearn.neighbors import NearestNeighbors

def neighbour_composition(X, species_labels, K, k=15, max_query=4000, seed=0):
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(seed)
    if X.shape[0] > max_query:
        qi = rng.choice(X.shape[0], size=max_query, replace=False)
    else:
        qi = np.arange(X.shape[0])
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, idx = nn.kneighbors(X[qi])
    idx = idx[:, 1:]
    neigh_sp = species_labels[idx]
    q_sp = species_labels[qi]
    mat = np.zeros((K, K), dtype=np.float64)
    for q_class in range(K):
        sel = q_sp == q_class
        if not np.any(sel):
            continue
        # per-query composition, then mean
        rows = neigh_sp[sel]
        for r in range(K):
            mat[q_class, r] = float((rows == r).mean())
    return mat

comp_raw = neighbour_composition(X_log, sp_s, n_species, k=K_NN,
                                 max_query=SCORECARD_CELLS, seed=SEED)
comp_z = neighbour_composition(z[umap_idx], sp_s, n_species, k=K_NN,
                               max_query=SCORECARD_CELLS, seed=SEED)
comp_ms = neighbour_composition(X_ms, species_labels[umap_idx_ms], n_species,
                                k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
comp_cs = neighbour_composition(X_cs, species_labels[umap_idx_ms], n_species,
                                k=K_NN, max_query=SCORECARD_CELLS, seed=SEED)
if compare_avail:
    comp_zc = neighbour_composition(run_c["final_state"]["z_mean"][umap_idx],
                                    sp_s, n_species, k=K_NN,
                                    max_query=SCORECARD_CELLS, seed=SEED)

# null: dataset frequencies in the query set
null = np.bincount(sp_s, minlength=n_species) / len(sp_s)

n_plots = 5 if compare_avail else 4
fig, axes = plt.subplots(1, n_plots, figsize=(4.2 * n_plots, 3.8),
                         sharey=True)
def plot_comp(ax, mat, title):
    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(n_species), species_names, rotation=30, ha="right")
    ax.set_yticks(range(n_species), species_names)
    ax.set_xlabel("neighbour species")
    ax.set_title(title)
    for i in range(n_species):
        for j in range(n_species):
            ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center",
                    color="white" if mat[i,j] > 0.6 else "black", fontsize=8)
    return im

plot_comp(axes[0], comp_raw, "raw log-counts")
axes[0].set_ylabel("query species")
plot_comp(axes[1], comp_z, f"z ({RUN_NAME})")
plot_comp(axes[2], comp_ms, "mu_shared (w/ batch)")
plot_comp(axes[3], comp_cs, "mu_cellstate (no batch)")
if compare_avail:
    plot_comp(axes[4], comp_zc, f"z ({COMPARE_RUN})")
fig.suptitle(f"§5 · kNN (k={K_NN}) neighbour composition — "
             f"null (dataset freq): {[f'{f:.2f}' for f in null]}", y=1.04)
if SAVE_FIGS:
    fig.savefig(f"figs/nb12_{RUN_NAME}_neighbour_mix.png", bbox_inches='tight')
plt.show()


## §6 — Interpretation (honest)

The scorecard + §5 matrices make one point concretely: **no representation
Chimera currently produces is species-invariant at the kNN level on close-
species, 1:1-batch data.** Raw, z (iter16), z (iter24 masked), `mu_shared_only`
with batch offset, and `mu_shared_cellstate` without batch offset all score
`knn_species_purity` at or very near 1.0 on Human + Monkey. The only
representation well below 1.0 would be a strong info-bottleneck on z
(smaller latent + much larger KL) — outside the round2 recipe space.

This is not a bug; it is the **round2 Caveat 34** architectural limit,
now decomposed empirically into three channels:

1. **Cell-type composition** — raw expression itself carries species
   because Human and Monkey have different cell-type fractions and
   species-specific baseline expression per cell type. `raw log-counts`
   already sits at ~99 % species purity.
2. **z-leakage** — the encoder takes `species_oh + batch_oh` and z
   retains species in nonlinearly-separable directions the linear
   adversarial can't remove. Visible on `z (iter16)` ≈ 1.00 even
   though the linear probe is at 0.74.
3. **Batch-offset injection** — `get_shared_only_expression()` adds the
   per-batch offset, which is 1:1 with species on this data and on every
   current Chimera dataset (E1.4 invariant). This is the `mu_shared_only`
   vs `mu_shared_cellstate` gap in the scorecard above.

Masking species+batch in the encoder (iter24) fixes the **linear**
channel and halves linear-probe accuracy (99 % → 72 %) but does not
close the kNN gap, because cell-type composition + nonlinear z-leakage
remain.

**Practical recommendation for downstream use.**
- **Don't use z or `mu_shared_only` as an integrated embedding** for
  kNN clustering or species-mixed UMAP visualisation. None of the
  current Chimera representations are species-free at kNN resolution.
- **Chimera's primary scientific output is per-gene α / Δα**, not the
  cell embedding. The 32-gate validation (notebook 10) + the α-biology
  analysis (notebook 13) are the panels that matter for the paper's
  scientific claim. This notebook is a failure-mode diagnostic, not a
  positive-result panel.
- If downstream species-mixed clustering is required, the honest path
  is a per-species analysis stratified by cell type, or an external
  integration tool (Harmony / scVI reference-mapping) applied *after*
  Chimera for clustering, while keeping Chimera's α / Δα for
  evolutionary inference.

**What this notebook is good for.**
- Visualising the cell-type structure preservation: `silhouette_celltype`
  is meaningfully higher on Chimera embeddings (0.22 / 0.29 / 0.40) than
  on raw (0.06) — biology IS preserved even when species mixing is not.
- Showing the batch-offset leakage channel is non-zero (gap between
  `mu_shared_only` and `mu_shared_cellstate` scorecard rows).
- Diagnostically: if a future run has `knn_celltype_purity` lower on z
  than on raw, that is a real regression and a bug — not expected.
